In [ ]:
"""import requests

# Replace the URL with one from the Available Endpoints
url = "https://api.sectors.app/v2/industries/"
api_key = "ff754b804e36ad701c40b38b48364d503da4b6d03eef8fa10b4dd3d2578cefc0"
headers = {"Authorization": api_key}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    data = response.json()
except requests.exceptions.HTTPError as err:
    raise SystemExit(err)"""

In [ ]:
"""import json

# Asumsi 'data_api' adalah variabel yang berisi response JSON dari Sectors API tadi
data_api = data

# 1. Menyimpan data ke dalam file 'berita_pasar.json'
with open('berita_pasar.json', 'w') as file:
    json.dump(data_api, file, indent=4)

print("Data berhasil disimpan ke berita_pasar.json!")

# 2. Cara membukanya kembali nanti:
with open('berita_pasar.json', 'r') as file:
    data_tersimpan = json.load(file)"""

Data berhasil disimpan ke berita_pasar.json!


In [33]:
import pandas as pd

data_sectors = pd.read_csv('sectors and subsectors.csv')
data_industri = pd.read_csv('industries.csv')
data_subindustri = pd.read_csv('subindustries.csv')

gabungan = pd.merge(data_sectors, data_industri, on='subsector',how='left')
gabungan = pd.merge(gabungan, data_subindustri, on='industry',how='left')
#print(gabungan.columns)

gabungan = gabungan.drop(columns=['Unnamed: 0_x', 'Unnamed: 0_y', 'Unnamed: 0'])

multiple_industry_sectors = gabungan.groupby(['sector', 'subsector', 'industry'])['sub_industry'].value_counts()

gabungan.to_csv('Sector_SubSector_Industry_SubIndustry.csv')


In [35]:
transport_logistic_company = pd.read_csv("Indonesia Stock Exchange Transportation and Logistics Tickers by Industry - Table 1.csv")
transport_logistic_company

,Ticker,Industry Name,Company Name,Sub-sector,Market Capitalization
0,GIAA,Airlines,PT Garuda Indonesia (Persero) Tbk Class B,Airlines / Code: K111,27.28 T IDR
1,JSMR,Transportation,PT Jasa Marga (Persero) Tbk Class B,Not in source,21.7 T IDR
2,TCPI,Transportation,PT Transcoal Pacific,Not in source,11.3 T IDR
3,CMNP,Transportation,PT Citra Marga Nusaphala Persada Tbk,Not in source,9.54 T IDR
4,RMKE,Transportation,PT RMK Energy Tbk,Not in source,9.23 T IDR
...,...,...,...,...,...
63,MIRA,Logistics & Deliveries,PT Mitra International Resources Tbk,Logistics & Deliveries / Code: K211,Not in source
64,PJHB,Logistics & Deliveries,PT Pelayaran Jaya Hidup Baru Tbk,Logistics & Deliveries / Code: K211,Not in source
65,RCCC,Logistics & Deliveries,PT Utama Radar Cahaya Tbk,Logistics & Deliveries / Code: K211,Not in source
66,WBSA,Logistics & Deliveries,PT BSA Logistics Indonesia Tbk,Logistics & Deliveries / Code: K211,Not in source


In [14]:
pd.set_option('display.max_rows', None)

In [181]:
df = pd.read_csv('../../dataset/csv/Top 10 transportation companies by market cap.csv')

In [171]:
import requests


def sectors_requester(endpoint, params=None):
  headers = {"Authorization": "8f78cc2a4fa85eb0606e28cf870f93d855d5f8a9b71afa51c2eec738c5147369"}
  url = "https://api.sectors.app/v2/"

  base_url = url + endpoint + "/"
  response = requests.get(base_url, headers=headers, params=params)
  response.raise_for_status()
  data = response.json()

  return data

In [172]:
#pulling corporate action data out of the api
import time
def pulling_data(ticker_list):
    response_list = []
    for symbol in ticker_list:
        endpoint = 'company/corporate-actions'
        endpoint = endpoint + f'/{symbol}'
        response = sectors_requester(endpoint)
        response_list.append(response)
        time.sleep(10)

    return response_list

In [ ]:
#turning corporate action result to dataframe
import pandas as pd
def turning_json_to_dataframe(response_list):
    event_types = list(response_list[0]['corporate_actions'])
    tables = {}
    for et in event_types:
        rows = []
        for entry in response_list:
            symbol = entry["symbol"]
            records = entry["corporate_actions"].get(et)
            if not records:
                continue
            for r in records:
                rows.append({"symbol": symbol, **r})
        tables[et] = pd.DataFrame(rows)

    long_df = pd.concat([df.assign(event_type=et) for et, df in tables.items() if not df.empty],axis=0, ignore_index=True)
    # 3. put the classifier columns up front for readability
    cols = ["symbol", "event_type"] + [c for c in long_df.columns if c not in ("symbol", "event_type")]
    long_df = long_df[cols]
    return long_df

In [ ]:
def companies_with_revenue_segments():
    endpoint = 'companies/list_companies_with_segments/'
    response = sectors_requester(endpoint)
    return response

In [177]:
companies_with_revenue_segments_list = companies_with_revenue_segments()

In [217]:
import time
def get_companies_revenue_segments(companies_with_revenue_segments_response, targeted_companies_ticker_list):
    list_of_response = []
    list_of_companies = list(companies_with_revenue_segments_response)

    for ticker in targeted_companies_ticker_list:
        if ticker in list_of_companies:
            endpoint = f'company/get-segments/{ticker}'
            list_of_financial_year = companies_with_revenue_segments_response[ticker]['financial_year']
            for financial_year in list_of_financial_year:
                params = {'financial_year' : financial_year}
                response = sectors_requester(endpoint, params)
                list_of_response.append(response)
                time.sleep(10)
    return list_of_response

In [218]:
response = get_companies_revenue_segments(companies_with_revenue_segments_list,df['symbol'])

In [ ]:
import pandas as pd
def scrapping_company_revenue_segment(company_revenue_segment_response):
    list_of_dataframe = []
    for industri in company_revenue_segment_response:
        ticker = industri['symbol']
        revenue_breakdown = pd.DataFrame(columns=['company_name','value', 'source', 'target'])
        company_name = []
        value = []
        source = []
        target = []
        for i in industri['revenue_breakdown'] : 
            company_name.append(ticker)
            value.append(i['value'])
            source.append(i['source'])
            target.append(i['target'])
        revenue_breakdown['company_name'] = company_name
        revenue_breakdown['value'] = value
        revenue_breakdown['source'] = source
        revenue_breakdown['target'] = target
        revenue_breakdown['financial_year'] = industri['financial_year']

        list_of_dataframe.append(revenue_breakdown)

    final_df = pd.concat(list_of_dataframe,axis=0)
    return final_df

In [250]:
#pulling shareholder composition data out of the api
import time
def pulling_data(ticker_list):
    response_list = []
    for symbol in ticker_list:
        endpoint = 'company/shareholders-composition'
        endpoint = endpoint + f'/{symbol}'
        current_year = time.localtime().tm_year
        for financial_year in [current_year-1, current_year]:
            print(financial_year)
            params = {'year' : financial_year}
            response = sectors_requester(endpoint, params)
            response_list.append(response)
            time.sleep(10)

    return response_list


In [251]:
response_list = pulling_data(df['symbol'])

2025
2026
2025
2026
2025
2026
2025
2026
2025
2026
2025
2026
2025
2026
2025
2026
2025
2026
2025
2026


In [ ]:
#company shareholder composition
list_of_dataframe = []
previous_ticker = ''
for i in range(len(response_list)):
    symbol = response_list[i]['symbol']
    dataframe = pd.DataFrame(response_list[i]['data'])
    dataframe['symbol'] = symbol
    if symbol == previous_ticker:
        dataframe = pd.concat([list_of_dataframe[len(list_of_dataframe)-1], dataframe], axis=0, ignore_index=True)
        list_of_dataframe[len(list_of_dataframe)-1] = dataframe
    else:
        previous_ticker = symbol
        previous_dataframe = dataframe
        list_of_dataframe.append(dataframe)
df_final = pd.concat(list_of_dataframe, axis=0, ignore_index=True)


In [ ]:
df_final.to_csv('../../dataset/csv/top10-transportation-by-market-cap-shareholder-composition.csv',index=False)